# 03 · Guardrails — 03 Fail Closed

**Everything here runs offline with no API key.** No model is called. Every failure
below is injected deliberately, and every dish, student and payload is synthetic.

**In → out:** a broken guard goes in — one that raises, and one that receives garbage.
Out comes the answer to the only question that matters about a guard's error path:
*when this guard cannot do its job, does it say no, or does it say yes?*

---

## Two gates from the same lab, failing in opposite directions

| | (a) `safe-bite` safety gate | (b) `terrier-ta` human-review gate |
|---|---|---|
| Guards | which dishes a person with allergies may be shown | whether a low-confidence grade gets released |
| On a normal run | excludes unsafe dishes with reasons | pauses and waits for an instructor |
| **On an error / malformed input** | returns **nothing** | returns **`{"approved": True}`** |
| Direction | **fails closed** | **fails open** |
| Status | correct | **a real defect, ported here unfixed** |

They were written by different people, for different products, and neither author
was being careless. The difference is a single decision about what a `try/except` or
an `isinstance` check does on the unhappy path — one line in each file — and that
single line is the difference between a guard and a decoration.

(b) is carried across **exactly as it exists in the donor**, defect intact. Fixing it
on the way in would delete the lesson: the point is that fail-open code does not look
wrong. It looks like a sensible default. You have to go looking for the error path to
see it.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `apply_safety_gate` | The working gate: excludes dishes on diet conflict and allergen hit, with a reason per exclusion. | 5 dishes in → 2 kept, 3 excluded |
| `_run_safety_gate` | The **fail-closed wrapper**. If the gate raises, returns `([], message)` — no dishes, not unfiltered dishes. | gate raises → `[]` |
| `gate_human_review_node` | The **fail-open defect**, ported verbatim. A non-dict resume payload becomes `{"approved": True}`. | `None` → approved |
| `apply_override_node` | The second fail-open, ported verbatim. `.get("approved", True)` — a *missing* key reads as approval. | `{}` → approved |
| `gate_human_review_node_closed` | A fail-closed rewrite, shown **alongside** the defect, not applied to it. | `None` → not approved |
| `error_path_verdict` | Runs every gate against every broken input and prints which direction each fails in. | a grid |

## Ported from

- (a) `safe-bite`'s `services/safety_gate.py` and the `_run_safety_gate` wrapper in
  `pipeline/orchestrator.py`.
- (b) `terrier-ta`'s `nodes/grading/gate_human_review.py` and
  `nodes/grading/apply_override.py`.

All dishes, allergen profiles, students and grades below are invented for this
notebook. No real menu, restaurant, student or grade data is used anywhere.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

In [ ]:
nbio.show_environment()
print("\nNo model is called in this notebook. Every failure below is injected on purpose.")

## Step 1 — (a) the gate that works, on a normal run

`apply_safety_gate` takes candidate dishes plus a diet type and an allergy list, and
splits them into `kept` and `excluded`. Two details in the donor are worth noticing
before the error path:

- **Every exclusion carries reasons and `checks_run`.** Not a bare filtered list — a
  `SafetyDecision` naming which checks ran and which fired. A silent filter cannot be
  audited after someone gets sick.
- **Allergens are matched against metadata *and* free text.** A dish whose
  `allergens` list is empty but whose description says "peanut" is still caught. The
  metadata is treated as incomplete by default, which is the right assumption about
  restaurant data.

`is_nonveg_text` is a one-line local stand-in for the donor's own helper, which lives
in a module of food-domain vocabulary that is not guardrail machinery.

In [ ]:
from dataclasses import dataclass, field, asdict
from typing import Any, Literal, Optional

GATE_VERSION = "1.0.0"

_NONVEG_WORDS = ("chicken", "mutton", "fish", "prawn", "egg", "lamb", "beef", "pork")


def is_nonveg_text(text: str) -> bool:
    low = (text or "").lower()
    return any(w in low for w in _NONVEG_WORDS)


@dataclass
class SafetyDecision:
    dish_id: str
    dish_name: str
    status: Literal["kept", "excluded"]
    reasons: list[str] = field(default_factory=list)
    checks_run: list[str] = field(default_factory=list)
    allergen_hits: list[str] = field(default_factory=list)
    diet_conflict: Optional[str] = None

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass
class SafetyGateResult:
    kept: list[dict[str, Any]]
    excluded: list[SafetyDecision]
    profile_snapshot: dict[str, Any]
    gate_version: str = GATE_VERSION

    def to_dict(self) -> dict[str, Any]:
        return {
            "kept": self.kept,
            "excluded": [e.to_dict() for e in self.excluded],
            "profile_snapshot": self.profile_snapshot,
            "gate_version": self.gate_version,
        }


def _diet_allows(item_type: str, diet_type: str) -> tuple[bool, Optional[str]]:
    t = (item_type or "").lower()
    d = (diet_type or "mix").lower()
    if d in {"mix", "any", "all", ""}:
        return True, None
    if d in {"veg", "vegetarian"} and t == "non-veg":
        return False, "diet:non-veg"
    if d in {"non-veg", "nonveg", "non_veg"} and t not in {"non-veg", "both"}:
        if t == "veg" or (not t and not is_nonveg_text("")):
            return False, "diet:veg"
    if d == "vegan" and t in {"non-veg", "veg", "both"}:
        return False, f"diet:vegan-conflict-{t}"
    return True, None


def _allergen_hits(item: dict[str, Any], allergies: list[str]) -> list[str]:
    if not allergies:
        return []
    name = (item.get("name") or "").lower()
    desc = (item.get("description") or "").lower()
    ing = (item.get("ingredients") or "").lower()
    meta = [a.lower() for a in (item.get("allergens") or [])]
    text = f"{name} {desc} {ing}"
    hits = []
    for allergy in allergies:
        a = allergy.lower().strip()
        if not a:
            continue
        if a in meta or a in text:
            hits.append(a)
    return hits


def apply_safety_gate(
    candidates: list[dict[str, Any]],
    *,
    diet_type: str = "mix",
    allergies: Optional[list[str]] = None,
) -> SafetyGateResult:
    allergies = allergies or []
    kept: list[dict[str, Any]] = []
    excluded: list[SafetyDecision] = []
    profile = {"diet_type": diet_type, "allergies": allergies}

    for idx, item in enumerate(candidates):
        dish_id = str(item.get("id") or item.get("name") or idx)
        name = item.get("name") or dish_id
        reasons: list[str] = []
        checks = ["diet_type", "allergen_metadata", "allergen_keyword"]
        diet_ok, diet_reason = _diet_allows(item.get("type") or item.get("diet_type") or "", diet_type)
        if not diet_ok and diet_reason:
            reasons.append(diet_reason)
        hits = _allergen_hits(item, allergies)
        if hits:
            reasons.extend([f"allergen:{h}" for h in hits])
        if reasons:
            excluded.append(
                SafetyDecision(
                    dish_id=dish_id,
                    dish_name=name,
                    status="excluded",
                    reasons=reasons,
                    checks_run=checks,
                    allergen_hits=hits,
                    diet_conflict=diet_reason,
                )
            )
        else:
            safe_item = dict(item)
            safe_item["safe"] = True
            safe_item["safety"] = {"status": "kept", "reasons": [], "checks_run": checks}
            kept.append(safe_item)

    return SafetyGateResult(kept=kept, excluded=excluded, profile_snapshot=profile)

## Step 2 — five invented dishes, one invented diner

The diner is vegetarian and allergic to peanut. The menu is written so that each
exclusion path fires at least once, including the one that catches a peanut the
restaurant's own `allergens` metadata forgot to list.

In [ ]:
MENU = [
    {"id": "d1", "name": "Paneer Tikka", "type": "veg",
     "description": "Grilled cottage cheese with spices", "allergens": ["dairy"]},
    {"id": "d2", "name": "Chicken Biryani", "type": "non-veg",
     "description": "Layered rice with chicken", "allergens": []},
    {"id": "d3", "name": "Peanut Chaat", "type": "veg",
     "description": "Crunchy salad", "allergens": ["peanut"]},
    {"id": "d4", "name": "Garden Salad", "type": "veg",
     "description": "Mixed greens with a light dressing", "allergens": []},
    {"id": "d5", "name": "Satay Skewers", "type": "veg",
     "description": "Served with a rich peanut sauce", "allergens": []},
]

DINER = {"diet_type": "veg", "allergies": ["peanut"]}

result = apply_safety_gate(MENU, diet_type=DINER["diet_type"], allergies=DINER["allergies"])

print(f"kept     : {[d['name'] for d in result.kept]}")
print(f"excluded : {[(e.dish_name, e.reasons) for e in result.excluded]}\n")

nbio.table(
    [(e.dish_name, ", ".join(e.reasons), e.diet_conflict or "-", ", ".join(e.allergen_hits) or "-")
     for e in result.excluded],
    headers=("dish", "reasons", "diet conflict", "allergen hits"),
)

kept_names = {d["name"] for d in result.kept}
assert kept_names == {"Paneer Tikka", "Garden Salad"}, kept_names
assert "Chicken Biryani" not in kept_names
assert "Peanut Chaat" not in kept_names

# d5 has an EMPTY allergens list. It is caught only because "peanut" appears in the
# free-text description -- the metadata alone would have let it through.
satay = next(e for e in result.excluded if e.dish_name == "Satay Skewers")
assert satay.allergen_hits == ["peanut"]
assert MENU[4]["allergens"] == [], "the metadata really was empty -- the text check is what caught it"
print("\nSatay Skewers: allergens metadata was empty; the free-text check caught it.")

## Step 3 — (a) the fail-closed wrapper, and the comment that explains it

Step 2 is the happy path. This is the part that matters. The donor wraps every call
to the gate in `_run_safety_gate`, whose docstring states the rule outright:

> Fails CLOSED: if the gate itself errors, no dishes are returned rather than
> silently falling back to an unfiltered list — a false "safe" is the worst possible
> bug, so an unknown gate outcome must never be treated as "safe by default".

Three structural choices to notice, all of them load-bearing:

1. **`return [], message` — not `return dishes`.** The tempting "degrade gracefully"
   move is to hand back the unfiltered list so the user still sees something. That
   move serves a peanut-allergic diner a peanut dish.
2. **The audit write is in its own `try`.** A logging failure must not change a
   decision that has already been made. Guard logic and guard telemetry fail
   independently.
3. **There is an outer `try` around the whole function.** Belt and suspenders: even a
   bug in the error handling itself still lands on `([], message)`.

In [ ]:
from typing import List, Dict


def _run_safety_gate(
    dishes: List[Dict[str, Any]],
    diet_type: str,
    allergies: List[str],
    *,
    gate_fn=apply_safety_gate,
    audit_fn=None,
) -> "tuple[List[Dict[str, Any]], Optional[str]]":
    """Apply the safety gate -- the SOLE hard-constraint enforcer.

    Every dish list handed to the agent must pass through here, no exceptions and no
    inline diet/allergy shortcuts elsewhere. Fails CLOSED: if the gate itself errors,
    no dishes are returned rather than silently falling back to an unfiltered list --
    a false "safe" is the worst possible bug, so an unknown gate outcome must never
    be treated as "safe by default".

    Returns (kept_dishes, error_message_or_None). When error_message is set, callers
    must surface it to the user instead of returning any dish from `dishes`.

    `gate_fn` and `audit_fn` are injection points added for this notebook, so the
    error paths below can be triggered without breaking the real gate.
    """
    fail_closed_message = (
        "Safety check is temporarily unavailable, so no dishes can be safely recommended "
        "right now. Please try again shortly."
    )
    try:
        gate_result = None
        gate_error: Optional[Exception] = None
        try:
            gate_result = gate_fn(dishes, diet_type=diet_type, allergies=allergies)
        except Exception as e:
            gate_error = e
            print(f"[ERROR] safety_gate failed — failing closed, returning no dishes: {e}")

        # Best-effort audit logging; a logging failure must never change the fail-closed
        # decision above (that decision is already made by this point).
        try:
            if audit_fn is not None:
                audit_fn(gate_result, gate_error, dishes)
        except Exception as log_err:
            print(f"[WARNING] failed to write safety run artifact (non-fatal): {log_err}")

        if gate_result is not None:
            return gate_result.kept, None
        return [], fail_closed_message
    except Exception as outer_err:
        # Belt-and-suspenders: any unexpected failure anywhere in this function must still
        # fail closed rather than propagate and risk an unfiltered dish list upstream.
        print(f"[ERROR] unexpected failure in _run_safety_gate — failing closed: {outer_err}")
        return [], fail_closed_message

## Step 4 — break it three different ways; it says no every time

Three separate injected failures, each hitting a different part of the wrapper:

| Failure | Where it lands |
|---|---|
| the gate raises | the inner `except` |
| the gate returns a malformed object | the attribute access, caught by the outer `except` |
| the audit write raises | the audit `try` — and must **not** change the verdict |

The third is the interesting one. The gate succeeded; only the logging broke. A
fail-closed design that also discarded good results whenever telemetry hiccupped
would be unusable, so the correct behaviour there is to keep the verdict — and the
donor's separate `try` is what makes that possible.

In [ ]:
def gate_that_raises(dishes, *, diet_type, allergies):
    raise RuntimeError("allergen table unavailable")


def gate_that_returns_garbage(dishes, *, diet_type, allergies):
    return "not a SafetyGateResult"


def audit_that_raises(gate_result, gate_error, dishes):
    raise IOError("disk full")


print("--- failure 1: the gate raises ---")
kept1, err1 = _run_safety_gate(MENU, "veg", ["peanut"], gate_fn=gate_that_raises)
print(f"kept: {kept1}  |  message: {err1[:44]}...\n")

print("--- failure 2: the gate returns something malformed ---")
kept2, err2 = _run_safety_gate(MENU, "veg", ["peanut"], gate_fn=gate_that_returns_garbage)
print(f"kept: {kept2}  |  message: {err2[:44]}...\n")

print("--- failure 3: only the audit write breaks ---")
kept3, err3 = _run_safety_gate(MENU, "veg", ["peanut"], audit_fn=audit_that_raises)
print(f"kept: {[d['name'] for d in kept3]}  |  message: {err3}\n")

# The gate broke: nothing is served, and the caller is told why.
assert kept1 == [] and err1
assert kept2 == [] and err2
# Not one dish leaked through a broken gate -- not even the two that ARE safe.
assert not any(d in kept1 + kept2 for d in MENU)
# The gate worked and only telemetry failed: the verdict stands.
assert {d["name"] for d in kept3} == {"Paneer Tikka", "Garden Salad"}
assert err3 is None

print("Broken gate -> 0 dishes served. Broken logger -> verdict unchanged.")

## Step 5 — the cost of failing closed, stated honestly

Failing closed is not free, and a notebook that pretended otherwise would be selling
something. When the gate breaks, the two dishes that were genuinely safe are withheld
too. The diner sees an error instead of a salad.

That is the trade, and it is only obviously correct because of what sits on the other
side of it. Compare the two ways to be wrong.

In [ ]:
nbio.table(
    [
        ("fails CLOSED", "a safe diner sees an error and eats elsewhere",
         "annoyed user, zero harm", "recoverable"),
        ("fails OPEN", "an allergic diner is served the peanut dish",
         "anaphylaxis", "not recoverable"),
    ],
    headers=("direction", "what happens when the gate breaks", "worst case", "recoverable?"),
)

print("\nBoth columns are real costs. They are not the same size, and they are not")
print("the same kind. One is an inconvenience; the other cannot be undone by an")
print("apology. That asymmetry -- not a preference for strictness -- is the argument.")

## Step 6 — (b) the fail-open defect, ported unfixed

Now the same lab's other gate. This one pauses a grading run when the model's
confidence is low, so an instructor can review before a grade is released.

Read line 21 of the donor. It is the last line of the function.

In [ ]:
# Ported from terrier-ta: nodes/grading/gate_human_review.py
#
# CAUTION, carried across honestly rather than silently fixed: the final line
# turns ANY non-dict resume payload -- None, a string, a list, a timeout that
# resolved to nothing -- into {"approved": True}. The one branch that means
# "I could not understand the instructor's decision" is the branch that grants
# approval. A low-confidence grade the gate was built to hold back is released
# by the gate's own error path.
#
# It is shown here, unaltered, because that is the lesson. The line does not
# look like a bug in review: `decision if isinstance(decision, dict) else <default>`
# is an ordinary defensive-coding shape, and the eye slides over the default.
# The direction of the default is the entire defect, and nothing in the line
# draws attention to it.
#
# Fixing it is a decision for whoever owns terrier-ta's grading pipeline, not
# for this port. Step 10 shows what the fix would look like, separately, without
# touching this function.

def _interrupt(payload):
    """Stand-in for langgraph.types.interrupt. The real one pauses the graph and
    returns whatever the instructor's resume call sends back. Here it returns a
    payload supplied by the test, so both the ordinary and broken cases are
    reachable offline."""
    return payload.get("_resume_with")


def gate_human_review_node(state: dict) -> dict:
    decision = _interrupt(
        {
            "gate": "human_review",
            "flagged_items": state.get("flagged_items") or [],
            "confidence": state.get("confidence") or {},
            "message": "Low-confidence grade — instructor review required.",
            "_resume_with": state.get("_resume_with"),
        }
    )
    return {**state, "override": decision if isinstance(decision, dict) else {"approved": True}}

## Step 7 — the malformed payload, producing approval

One synthetic student, one low-confidence grade, five different resume payloads. The
first is what the instructor is supposed to send. The other four are the ways a
resume goes wrong in practice: a timeout resolving to `None`, a client sending a bare
string, a client sending a list, and a boolean `False` that a reader would swear means
"not approved".

Watch the `False` row in particular.

In [ ]:
FLAGGED_STATE = {
    "run_id": "synthetic-run-001",
    "student": "Student A",
    "raw_grade": {"score": 42, "max": 100},
    "confidence": {"overall": 0.31},
    "flagged_items": ["q3: answer may be off-topic", "q5: rubric match unclear"],
}

PAYLOADS = [
    ("instructor declines", {"approved": False, "override_by": "instructor"}),
    ("resume timed out (None)", None),
    ("client sent a string", "approved"),
    ("client sent a list", [{"approved": False}]),
    ("client sent False", False),
]

rows = []
for label, payload in PAYLOADS:
    out = gate_human_review_node({**FLAGGED_STATE, "_resume_with": payload})
    override = out["override"]
    approved = bool(override.get("approved")) if isinstance(override, dict) else None
    rows.append((label, repr(payload)[:26], repr(override)[:44], "APPROVED" if approved else "held"))

nbio.table(rows, headers=("resume payload", "value", "resulting override", "outcome"))

# The instructor's real decline is honoured.
assert gate_human_review_node({**FLAGGED_STATE, "_resume_with": PAYLOADS[0][1]})["override"]["approved"] is False

# Every malformed payload -- including an explicit False -- becomes approval.
for label, payload in PAYLOADS[1:]:
    out = gate_human_review_node({**FLAGGED_STATE, "_resume_with": payload})
    assert out["override"] == {"approved": True}, (label, out["override"])

print("\nFour of five payloads that are NOT an instructor approval produce approval.")
print("`False` is the worst of them: it is not a dict, so it takes the else branch")
print("and is converted into its own opposite.")

## Step 8 — a second fail-open, one node downstream

The gate hands its `override` to `apply_override_node`, which decides whether the run
continues. It has the same problem in a different disguise — not an `isinstance`
default this time, but a dictionary `.get` default.

`override.get("approved", True)`: a payload that simply **omits** the `approved` key
is treated as approved. An empty dict `{}` — the shape a partially-constructed or
schema-mismatched payload most often takes — sails straight through.

In [ ]:
# Ported from terrier-ta: nodes/grading/apply_override.py
#
# CAUTION, carried across honestly rather than silently fixed: the default in
# `override.get("approved", True)` means a payload with NO `approved` key is
# treated as approved. Combined with Step 6's node, there are now two independent
# routes to an unreviewed grade being released: a payload that is not a dict, and
# a dict that is missing one key.
#
# Ported unaltered for the same reason as Step 6 -- `.get(key, True)` is an
# everyday line, and the defect is entirely in the choice of default.

def _write_json(*args, **kwargs):
    """Stand-in for the donor's artifact write. Does nothing here."""


def apply_override_node(state: dict) -> dict:
    override = state.get("override")
    if isinstance(override, dict) and override.get("grade"):
        state = {**state, "raw_grade": override["grade"]}
    elif not (isinstance(override, dict) and override.get("approved", True)):
        errors = list(state.get("errors") or [])
        errors.append("human_gate:review_required")
        return {**state, "errors": errors}
    saved_to = state.get("saved_to") or ""
    if saved_to and override:
        _write_json(saved_to, "override", override)
    return state


def released(state: dict) -> bool:
    """A grade is released if the run carries no review-required error."""
    return "human_gate:review_required" not in (state.get("errors") or [])


OVERRIDES = [
    ("explicit decline", {"approved": False}),
    ("empty dict", {}),
    ("key misspelled", {"aproved": False}),
    ("wrong type for the key", {"approved": "no"}),
    ("no override at all", None),
]

rows = []
for label, override in OVERRIDES:
    out = apply_override_node({**FLAGGED_STATE, "override": override})
    rows.append((label, repr(override)[:24], "RELEASED" if released(out) else "held"))

nbio.table(rows, headers=("override", "value", "grade"))

assert not released(apply_override_node({**FLAGGED_STATE, "override": {"approved": False}}))
assert released(apply_override_node({**FLAGGED_STATE, "override": {}})), \
    "an empty dict is treated as approval -- this is the defect"
assert released(apply_override_node({**FLAGGED_STATE, "override": {"aproved": False}})), \
    "a typo'd key is treated as approval"
assert released(apply_override_node({**FLAGGED_STATE, "override": {"approved": "no"}})), \
    "the string 'no' is truthy, so it reads as approval"

print("\nA typo in a key name releases an unreviewed grade. So does the string \"no\".")

## Step 9 — end to end: the two nodes together

Neither defect needs the other. But they compose, and the composition is what a real
run does: the gate produces an override, the next node acts on it. A timed-out resume
goes in at one end; a released, never-reviewed grade comes out the other, with no
error recorded anywhere and nothing in the state to indicate that a human was
supposed to look at it.

In [ ]:
def run_grading_gate(resume_payload) -> dict:
    gated = gate_human_review_node({**FLAGGED_STATE, "_resume_with": resume_payload})
    return apply_override_node(gated)


final = run_grading_gate(None)   # the instructor never responded

print(f"confidence      : {final['confidence']['overall']}  (low -- this is why the gate fired)")
print(f"flagged items   : {len(final['flagged_items'])}")
print(f"override        : {final['override']}")
print(f"errors          : {final.get('errors') or '(none)'}")
print(f"grade released  : {released(final)}")
print(f"score published : {final['raw_grade']}")

assert final["override"] == {"approved": True}
assert released(final), "the whole pipeline releases an unreviewed grade on a timeout"
assert not final.get("errors"), "and records nothing to indicate it happened"

print("\nNo human ever saw this grade. Nothing in the final state says so.")

## Step 10 — what the fix would look like (shown, not applied)

For completeness, and kept deliberately separate from the ported functions above so
the port stays faithful. The change is small in both nodes and identical in shape:
**make the unrecognized branch deny.**

In [ ]:
def gate_human_review_node_closed(state: dict) -> dict:
    """Fail-closed variant. NOT applied to the ported node above -- shown so the
    size of the change is visible. An unreadable decision is not a decision."""
    decision = _interrupt({**state, "_resume_with": state.get("_resume_with")})
    if not isinstance(decision, dict):
        # The only honest reading of an unparseable resume: nobody approved anything.
        return {**state, "override": {"approved": False, "reason": "unparseable_resume_payload"}}
    return {**state, "override": decision}


def apply_override_node_closed(state: dict) -> dict:
    """Fail-closed variant of the downstream node: `approved` must be present and
    must be exactly True."""
    override = state.get("override")
    if isinstance(override, dict) and override.get("grade"):
        state = {**state, "raw_grade": override["grade"]}
    elif not (isinstance(override, dict) and override.get("approved") is True):
        errors = list(state.get("errors") or [])
        errors.append("human_gate:review_required")
        return {**state, "errors": errors}
    return state


for payload in (None, False, "approved", [{"approved": False}]):
    out = apply_override_node_closed(gate_human_review_node_closed({**FLAGGED_STATE, "_resume_with": payload}))
    assert not released(out), payload

for override in ({}, {"aproved": False}, {"approved": "no"}, None):
    assert not released(apply_override_node_closed({**FLAGGED_STATE, "override": override}))

# The legitimate approval still works -- failing closed must not mean failing always.
ok = apply_override_node_closed(
    gate_human_review_node_closed({**FLAGGED_STATE, "_resume_with": {"approved": True, "override_by": "instructor"}})
)
assert released(ok)

print("fail-closed variant: every malformed payload holds the grade;")
print("a genuine instructor approval still releases it.")
print("\nDiff in the first node:  else {\"approved\": True}  ->  else {\"approved\": False, ...}")
print("Diff in the second node: .get(\"approved\", True)     ->  .get(\"approved\") is True")

## Step 11 — every gate, every broken input, one grid

The whole notebook in one table. Same kind of input — something the guard cannot
make sense of — put through each guard, printing which direction it falls.

In [ ]:
def error_path_verdict() -> list[tuple]:
    rows = []

    kept, msg = _run_safety_gate(MENU, "veg", ["peanut"], gate_fn=gate_that_raises)
    rows.append(("safe-bite  _run_safety_gate", "gate raises",
                 f"{len(kept)} dishes served", "CLOSED", "correct"))

    kept, msg = _run_safety_gate(MENU, "veg", ["peanut"], gate_fn=gate_that_returns_garbage)
    rows.append(("safe-bite  _run_safety_gate", "gate returns garbage",
                 f"{len(kept)} dishes served", "CLOSED", "correct"))

    out = gate_human_review_node({**FLAGGED_STATE, "_resume_with": None})
    rows.append(("terrier-ta gate_human_review", "resume is None",
                 str(out["override"]), "OPEN", "DEFECT"))

    out = gate_human_review_node({**FLAGGED_STATE, "_resume_with": False})
    rows.append(("terrier-ta gate_human_review", "resume is False",
                 str(out["override"]), "OPEN", "DEFECT"))

    out = apply_override_node({**FLAGGED_STATE, "override": {}})
    rows.append(("terrier-ta apply_override", "override is {}",
                 "released" if released(out) else "held", "OPEN", "DEFECT"))

    out = apply_override_node_closed(gate_human_review_node_closed({**FLAGGED_STATE, "_resume_with": None}))
    rows.append(("fail-closed rewrite (Step 10)", "resume is None",
                 "released" if released(out) else "held", "CLOSED", "shown, not applied"))

    return rows


rows = error_path_verdict()
nbio.table(rows, headers=("guard", "broken input", "result", "fails", "status"))

directions = [r[3] for r in rows]
assert directions.count("CLOSED") == 3
assert directions.count("OPEN") == 3

## Step 12 — the rule

Stated once, plainly, because it is the only thing from this notebook worth
memorizing:

> **When a guard errors, the safe default is to deny. Any guard whose error path
> grants permission is a guard in name only.**

It follows from what a guard is *for*. A guard exists because some outcomes are worse
than others; that asymmetry is the entire reason to spend code on it. An error path
that grants permission throws the asymmetry away at exactly the moment it is needed —
when the system has stopped understanding its own state. A guard that permits when
confused does not reduce the probability of the bad outcome. It relocates it to the
cases nobody tested.

Three practical consequences:

1. **Write the error path first, then the happy path.** The unhappy branch is where
   the guarantee lives, and it is the branch nobody reviews.
2. **Treat "unrecognized" as a third value, not as a synonym for "yes".** `None`,
   `{}`, `False`, a misspelled key and a string are all *absence of a decision*.
   Absence of a decision is not approval.
3. **Test the guard by breaking it, not by using it.** Every fail-open case here is
   invisible to a test suite that only exercises well-formed input. Steps 7 and 8
   would pass a full happy-path suite without a murmur.

## Wrap-up

Two gates, one lab. One was wrapped in a `try` whose except clause returns nothing and
a docstring that says why. The other ends in `else {"approved": True}` and hands it to
a node that reads a missing key as consent. Both look like careful code. Only one of
them is a guard.

The fail-open node is still in `terrier-ta` and is still shipped that way as of this
port. It is reproduced here without correction — an honest defect carried across is a
bounded first contribution for somebody, and a silently fixed one is a lesson deleted.

The full stage, in one line each:

- `01-input-guard.ipynb` — a guard that runs after the expensive thing is a report.
- `02-tool-guard.ipynb` — stop the action, and prove the action's body never ran.
- `03-fail-closed.ipynb` — and when the guard itself breaks, it must say no.

## What did not come across

- **The real LangGraph `interrupt` and the resume path.** `_interrupt` here is four
  lines that return a payload the test supplies. The real node suspends a graph
  against a checkpointer and is resumed by a separate HTTP call, possibly days later
  — which is exactly why the "resume payload is malformed" case is common enough to
  matter, and exactly what cannot be shown offline.
- **`resume_grade_override`.** The donor's service that constructs the resume
  payload, writes an audit row and restarts the thread. Its own signature defaults to
  `approved: bool = True`, which is a third instance of the same pattern — named here,
  not ported, because two demonstrations of one defect is already one more than the
  lesson needs.
- **`filter_dishes_by_diet` / `filter_dishes_by_allergy`.** The donor's
  `filter_dish_names_by_safety` path, which filters by dish *name* only. Food-domain
  vocabulary, not guardrail machinery.
- **The S3 audit write.** `_run_safety_gate`'s real artifact write goes to object
  storage. Replaced with an injectable `audit_fn` so Step 4 can break the logger
  without breaking anything else.
- **Whether `terrier-ta`'s defect has ever caused a wrong grade in practice.** Not
  established here. What is established is that the code path exists and that four of
  five malformed payloads reach it. Frequency is a separate investigation, and
  claiming a number for it without one would be inventing evidence.